# 04 -- Stage 4: Add Synthetic Playground Data

Batch-generates synthetic manipulations over real images using the
same `services/playground/controller.py` the live app uses, appends
them to the training manifest, and retrains.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import cv2, random, pandas as pd
from pathlib import Path
from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.playground.controller import generate

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
images_dir = dataset_root / settings.data.images_dir
synthetic_dir = resolve_path(settings.data.synthetic_dir)
synthetic_dir.mkdir(parents=True, exist_ok=True)

manipulation_types = ["copy_move", "splicing", "compression", "blur", "beautify"]
rows = []

for image_path in images_dir.glob("*.jpg"):
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    manipulation_type = random.choice(manipulation_types)
    result = generate(manipulation_type, image)

    out_name = f"{image_path.stem}_{manipulation_type}.jpg"
    out_path = synthetic_dir / out_name
    cv2.imwrite(str(out_path), result.image)
    rows.append({"filename": str(out_path), "label": result.label, "type": result.manipulation_type})

synthetic_manifest = resolve_path("data/synthetic_labels.csv")
pd.DataFrame(rows).to_csv(synthetic_manifest, index=False)
print("Generated", len(rows), "synthetic samples ->", synthetic_manifest)

## Persist to Drive

`synthetic_dir` and `synthetic_manifest` above are written relative to
`PROJECT_ROOT` (this Colab session's local disk), **not** `dataset_root`
-- even if your real images were read from Drive, this output is local
and vanishes when the session ends unless copied over now.

In [ ]:
import shutil

if IS_COLAB:
    drive_data_dir = Path(settings.data.root_drive)
    drive_data_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(synthetic_dir, drive_data_dir / settings.data.synthetic_dir, dirs_exist_ok=True)
    shutil.copy(synthetic_manifest, drive_data_dir / "synthetic_labels.csv")
    print("Copied synthetic_data/ and synthetic_labels.csv to", drive_data_dir)
else:
    print("Not on Colab -- skipping Drive copy.")

Concatenate `data/synthetic_labels.csv` onto your Stage 3 merged
manifest and rerun training from the latest checkpoint.